In [ ]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----
# ! pip install -r requirements.txt
# ! pip list
# ! conda list

# !conda install -y requests

import os
import sys
import glob
from datetime import datetime

import re
import time
import math

import urllib.parse
import requests

import pandas as pd

from tqdm import tqdm  # status bars

# base settings
# -----
conf_vlab_name     = "NIS"
# conf_workflow_name = "Marine"

# conf_workflow_id   = f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
param_workflow_name = "workflow name"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# conf_dir_code = os.path.join("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, "Git public", "library")
# if not os.path.exists(conf_dir_code):
#     os.makedirs(conf_dir_code)

# conf_dir_data  = os.path.join("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, param_workflow_name)
# if not os.path.exists(conf_dir_data):
#     os.makedirs(conf_dir_data)

# local
# -----
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp = os.path.join("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      = "naa-vre-public"
conf_minio_public_bucket_root = f"vl-{conf_vlab_name.lower()}"
conf_minio_public_local_root  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data")

conf_minio_user_bucket        = "naa-vre-user-data"
# conf_minio_user_bucket_root   = param_user_email
conf_minio_user_bucket_root   = conf_vlab_name
conf_minio_user_local_root    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   param_workflow_name)
conf_minio_user_local_flog    = os.path.join(conf_minio_user_local_data, "log.md")

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# Input param
# -----

# USER file paths & parameters
# .....
# -- Input
conf_delimiter_tsv = "\t"
conf_delimiter_csv = ","

param_fname_locations_tsv = "input-locations.tsv"  # your uploaded locations file
param_fname_species_tsv   = "input-species.tsv"  # your uploaded species file
# -- Output

# -- Parameters
param_MAX_OBIS_RECORDS       = 10000  # how many OBIS occurrences to consider
param_MAX_REGION_DIAGONAL_KM = 3000   # filter out marine regions larger than this

print("Finish: NaaVRE parameters")
print(f"Workspace public:")
print(f"  Root: {conf_minio_public_local_root}")
print(f"  Code: {conf_minio_public_local_code}")
print(f"  Data: {conf_minio_public_local_data}")

print(f"Workspace user:")
print(f"  Root: {conf_minio_user_local_root}")
print(f"  Code: {conf_minio_user_local_code}")
print(f"  Data: {conf_minio_user_local_data}")
print(f"  Log:  {conf_minio_user_local_flog}")


In [ ]:
# DNA, workflow start
# ---
# NaaVRE:
#  cell:
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

# prepare folders
# .....
if not os.path.exists(conf_dir_data_local_tmp):
    os.makedirs(conf_dir_data_local_tmp)

# if not os.path.exists(conf_minio_public_local_root):
#     os.makedirs(conf_minio_public_local_root)

if not os.path.exists(conf_minio_user_local_root):
    os.makedirs(conf_minio_user_local_root)

if not os.path.exists(conf_minio_user_local_data):
    os.makedirs(conf_minio_user_local_data)
    
with open(conf_minio_user_local_flog, "w+") as fp_log:
    fp_log.write(f"# {param_workflow_name}\n")

# create log
# .....
print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Start"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n")
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n")

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)
# print("sys.path minio_public")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# py_module_name = 'nest_asyncio_naavre'
# if py_module_name in sys.modules:
#     print(f"{py_module_name} already in sys.modules")
# elif (spec := importlib_util.find_spec(py_module_name)) is not None:
#     # If you chose to perform the actual import ...
#     py_module_obj = importlib_util.module_from_spec(spec)
#     sys.modules[py_module_name] = py_module_obj
#     spec.loader.exec_module(py_module_obj)
#     print(f"{py_module_name} has been imported")
# else:
#     print(f"can't find the {py_module_name} module")
# py_module_obj.apply()

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)
# print("sys.path minio_user")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")


In [ ]:
# NIS, Calculation
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Main Calculation"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
sys.path.append(conf_minio_public_local_code)
print("sys.path minio_public")
for tmp_path in sys.path: print(f"* {tmp_path}")

py_module_name = 'nis_classify_invasiveness.'
if py_module_name in sys.modules:
    print(f"{py_module_name} already in sys.modules")
elif (spec := importlib_util.find_spec(py_module_name)) is not None:
    # If you chose to perform the actual import ...
    py_module_obj = importlib_util.module_from_spec(spec)
    sys.modules[py_module_name] = py_module_obj
    spec.loader.exec_module(py_module_obj)
    print(f"{py_module_name} has been imported")
else:
    print(f"can't find the {py_module_name} module")
py_module_obj.apply()

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)
# print("sys.path minio_user")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# input
# -----
dummy_cell_arg_i = "dummy input"

print(param_MAX_OBIS_RECORDS)
print(param_MAX_REGION_DIAGONAL_KM)

param_file_locations_tsv = os.path.join(conf_minio_user_local_root, param_fname_locations_tsv)
param_file_species_tsv   = os.path.join(conf_minio_user_local_root, param_fname_species_tsv)

# output
# -----
dummy_cell_arg_o = "dummy output"

fname_result_tsv = "output-results.tsv"
file_output_tsv  = os.path.join(conf_minio_user_local_data, fname_result_tsv)

# func
# -----
def classify_with_progress(scientific_name, dict_locations, max_diagonal_km):
    """
    Run WRiMS classification for each (lat, lon) in `locations`

    dict_locations: dict[label] = (lat, lon)

    Returns dict[label] = {
        "mrgids": [...],
        "status": {...},
        "label": "human-friendly label"
    }
    """
    results = {}

    # for key in tqdm(dict_locations.keys(),
    #                 desc=f"WRiMS status for {scientific_name}",
    #                 unit="location",
    #                 leave=False):
    for key in dict_locations.keys():
        lat, lon = dict_locations[key]
        
        mrgids = get_filtered_mrgids_for_coord(
            lat,
            lon,
            max_diagonal_km
        )
        status = classify_invasiveness(scientific_name, mrgids)
        
        results[key] = {
            "mrgids": mrgids,
            "status": status,
            "label":  wrims_label(status)
        }

    return results

# start
# -----
rtn_results = []

# Main-1 Read input TSVs
# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = "## 1 Read input TSVs"
    print(str_print)
    fp_log.write(f"{str_print}\n")

df_locations = pd.read_csv(param_file_locations_tsv, sep=conf_delimiter_tsv)
df_species   = pd.read_csv(param_file_species_tsv,   sep=conf_delimiter_tsv)
    
# Main-2 Convert DMS to decimal degrees
# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = "## 2 Convert DMS to decimal degrees"
    print(str_print)
    fp_log.write(f"{str_print}\n")

df_locations["decimalLatitude"]  = df_locations["verbatimLatitude"].apply(dms_to_decimal)
df_locations["decimalLongitude"] = df_locations["verbatimLongitude"].apply(dms_to_decimal)
    
# Main-3 Join species with df_locations via locationID
# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = "## 3 Join species with df_locations via locationID"
    print(str_print)
    fp_log.write(f"{str_print}\n")

df_merged = df_species.merge(df_locations, on="locationID", how="inner")

str_print = f"> Total (location, species) cases to analyse: {len(df_merged)}"
print(str_print)
fp_log.write(f"{str_print}\n")

# Main-4 Loop over each row and run your full analysis
# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = "## 4 Loop over each row and run your full analysis"
    print(str_print)
    fp_log.write(f"{str_print}\n")

    # for _, row in tqdm(df_merged.iterrows(),
    #                     total=len(df_merged),
    #                     desc="Total analyses",
    #                     unit="case"):
    for _, row in df_merged.iterrows():
        row_loc_id     = row["locationID"]
        row_verb_lat   = row["verbatimLatitude"]
        row_verb_lon   = row["verbatimLongitude"]
        row_lat        = float(row["decimalLatitude"])
        row_lon        = float(row["decimalLongitude"])
        row_sci_name   = row["scientificName"]
        row_taxon_rank = row.get("taxonRank", "")
    
        str_print = f"* {row_loc_id}, {row_sci_name}"
        print(str_print)
        fp_log.write(f"{str_print}\n")

        # ---- OBIS: closest occurrence by sea-only distance ----
        try:
            closest = find_closest_obis_occurrence(
                row_sci_name,
                row_lat,
                row_lon,
                limit=param_MAX_OBIS_RECORDS
            )
            obis_lat     = closest["decimalLatitude"]
            obis_lon     = closest["decimalLongitude"]
            obis_dist_km = closest["sea_distance_km"]
    
        except Exception as e:
            str_print = f"\n```\n[WARN] OBIS step failed for {row_sci_name} at {row_loc_id}: {e}\n```\n"
            print(str_print)
            fp_log.write(f"{str_print}\n")
            
            closest      = None
            obis_lat     = None
            obis_lon     = None
            obis_dist_km = None
    
        # ---- WRiMS classification for new & closest OBIS df_locations ----
        locations_dict = {
            "new_observation": (row_lat, row_lon)
        }
        if closest is not None:
            locations_dict["closest_obis"] = (obis_lat, obis_lon)
    
        wrims_results = classify_with_progress(row_sci_name, locations_dict, param_MAX_REGION_DIAGONAL_KM)
    
        new_label  = wrims_results["new_observation"]["label"]
        obis_label = wrims_results.get("closest_obis", {}).get("label")
    
        # ---- Collect row for output ----
        rtn_results.append(
            {
                "locationID":                row_loc_id,
                "verbatimLatitude":          row_verb_lat,
                "verbatimLongitude":         row_verb_lon,
                "decimalLatitude":           row_lat,
                "decimalLongitude":          row_lon,
                "scientificName":            row_sci_name,
                "taxonRank":                 row_taxon_rank,
                "closest_obis_lat":          obis_lat,
                "closest_obis_lon":          obis_lon,
                "distance_over_water_km":    obis_dist_km,
                "wrims_status_new_location": new_label,
                "wrims_status_closest_obis": obis_label
            }
        )
    
# Main-5 Write single TSV output
# .....
with open(conf_minio_user_local_flog, "a+") as fp_log:
    str_print = "## 5 Write single TSV output"
    print(str_print)
    fp_log.write(f"{str_print}\n")

df_out = pd.DataFrame(rtn_results)
df_out.to_csv(file_output_tsv, sep=conf_delimiter_tsv, index=False)
    
# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {file_output_tsv}\n")

print(f"Finish: {workflow_step}")


In [ ]:
# DNA, workflow finish
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
# ...

import os
import sys
from datetime import datetime

# sys.path.append(conf_minio_public_local_code)
# sys.path.append(conf_minio_user_local_code)

print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Finish"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)
# print("sys.path minio_public")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# py_module_name = 'nest_asyncio_naavre'
# if py_module_name in sys.modules:
#     print(f"{py_module_name} already in sys.modules")
# elif (spec := importlib_util.find_spec(py_module_name)) is not None:
#     # If you chose to perform the actual import ...
#     py_module_obj = importlib_util.module_from_spec(spec)
#     sys.modules[py_module_name] = py_module_obj
#     spec.loader.exec_module(py_module_obj)
#     print(f"{py_module_name} has been imported")
# else:
#     print(f"can't find the {py_module_name} module")
# py_module_obj.apply()

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)
# print("sys.path minio_user")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")
